# Schritt 1: Transfer Learning mit VGG19

In diesem Notebook erkunden wir Transfer Learning mit VGG19 in zwei Phasen:

**Phase 1 – VGG19 kennenlernen**: Das vollständige VGG19-Modell laden und
ein eigenes Bild mit den originalen 1.000 ImageNet-Klassen klassifizieren.

**Phase 2 – Transfer Learning vorbereiten**: Den Klassifikationskopf entfernen,
Gewichte einfrieren und einen eigenen Kopf für eine neue Aufgabe anhängen.

## Lernziele

Nach diesem Notebook sollten Sie:

- erklären können, was `include_top=False` bewirkt
- verstehen, warum und wie Gewichte eingefroren werden
- den Anteil trainierbarer vs. eingefrorener Parameter benennen und interpretieren können

## Leitfragen (zum Nachdenken beim Durcharbeiten)

- Warum ist `include_top=False` so wichtig für Transfer Learning?
- Was bewirkt das Einfrieren der Gewichte – und was würde ohne Einfrieren passieren?
- Wie viele Parameter werden tatsächlich trainiert vs. wie viele gibt es insgesamt?
- Warum erreicht Transfer Learning mit wenigen Bildern trotzdem gute Ergebnisse?

# Schritt 2: Pfad-Konfiguration
## Bilddatei zentral definieren

Passen Sie den Pfad zum Beispielbild hier an.

In [ ]:
SAMPLE_IMAGE = "/content/drive/MyDrive/Colab Notebooks/DHBW Sommersemester 2026/Data/teddy.png"

IMG_SIZE = (224, 224)   # Feste Eingabegröße von VGG19

print(f"Bild:      {SAMPLE_IMAGE}")
print(f"Bildgröße: {IMG_SIZE}")

# Schritt 3: Bibliotheken importieren

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

# Schritt 4: Google Drive einbinden
## Google Drive mounten

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ══════════════════════════════════════════════════════
# PHASE 1: VGG19 kennenlernen – Inferenz mit vollem Modell
# ══════════════════════════════════════════════════════

# Schritt 5: Vollständiges VGG19-Modell laden
## VGG19 mit Klassifikationskopf (`include_top=True`)

Wir laden das vollständige VGG19-Modell inklusive des originalen
Klassifikationskopfes für 1.000 ImageNet-Klassen.

Beobachtungsauftrag:
- Wie viele Schichten hat das Modell?
- Was ist die letzte Schicht – und warum hat sie 1.000 Ausgaben?

In [ ]:
model_full = tf.keras.applications.VGG19(weights="imagenet", include_top=True)
model_full.summary()
# ▶ Letzte Schicht: Dense(1000, softmax) → 1000 ImageNet-Klassen

# Schritt 6: Eigenes Bild mit VGG19 klassifizieren
## Inferenz: Was erkennt VGG19 in unserem Bild?

Wir laden das Beispielbild und lassen VGG19 eine Vorhersage treffen.

In [ ]:
img = tf.keras.utils.load_img(SAMPLE_IMAGE, target_size=IMG_SIZE)
img_array = tf.keras.utils.img_to_array(img)

plt.imshow(img_array.astype("uint8"))
plt.title("Eingabebild")
plt.axis("off")
plt.show()

# Vorverarbeitung für VGG19 (Normierung + BGR-Konvertierung)
img_preprocessed = tf.keras.applications.vgg19.preprocess_input(img_array)
pred = model_full.predict(img_preprocessed.reshape(1, 224, 224, 3))

top5 = tf.keras.applications.vgg19.decode_predictions(pred, top=5)
print("Top-5-Vorhersagen des vortrainierten VGG19:")
for _, name, prob in top5[0]:
    print(f"  {name:30s}: {prob*100:.2f}%")

# Schritt 7: Beobachtung Phase 1
## Zwischenfazit

VGG19 klassifiziert bereits sehr gut – aber nur aus 1.000 ImageNet-Klassen.

Für eine **eigene Aufgabe** brauchen wir einen angepassten Klassifikationskopf.
Das ist der Kern von Transfer Learning – und genau das bauen wir in Phase 2.

# ══════════════════════════════════════════════════════
# PHASE 2: Transfer Learning – VGG19 anpassen
# ══════════════════════════════════════════════════════

# Schritt 8: VGG19 OHNE Klassifikationskopf laden
## `include_top=False`

Mit `include_top=False` entfernen wir die letzten Dense-Schichten.

Was bleibt: der **Convolutional-Teil** – ein universeller Feature-Extraktor,
der Kanten, Texturen und abstrakte Muster erkennt.

Diese Fähigkeiten sind aufgabenunabhängig und auf neue Probleme übertragbar.

In [ ]:
base_model = tf.keras.applications.VGG19(
    weights="imagenet",
    include_top=False,         # ← Klassifikationskopf weglassen
    input_shape=(224, 224, 3)
)
base_model.summary()
# ▶ Letzte Schicht: MaxPooling2D – kein Dense mehr

# Schritt 9: Gewichte einfrieren (Feature Extraction)
## Warum einfrieren?

Die VGG19-Gewichte kodieren visuelles Wissen aus 1,2 Mio. Trainingsbildern.
Wir wollen dieses Wissen **behalten** und nur unseren neuen Kopf trainieren.

Ohne Einfrieren würde das Netz das Vorwissen mit einem kleinen Datensatz
überschreiben – bekannt als **Catastrophic Forgetting**.

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

trainable_params   = sum(tf.size(w).numpy() for w in base_model.trainable_weights)
untrainable_params = sum(tf.size(w).numpy() for w in base_model.non_trainable_weights)
print(f"Trainierbare Parameter (Basis):   {trainable_params:>10,}")
print(f"Eingefrorene Parameter (Basis):   {untrainable_params:>10,}")

# Schritt 10: Eigenen Klassifikationskopf hinzufügen
## Neuer Kopf für unsere Aufgabe

- `GlobalAveragePooling2D` → reduziert Feature-Maps auf einen Vektor
- `Dense(128, relu)` → lernt aufgabenspezifische Kombinationen
- `Dense(1, sigmoid)` → Binärklassifikation: z. B. Teddybär ja / nein

In [ ]:
inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(128, activation="relu")(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_tl = tf.keras.Model(inputs, outputs)
model_tl.summary()
# ▶ Trainable params ≪ Non-trainable params

# Schritt 11: Manuelle Parameterberechnung
## Parameter des neuen Kopfes nachrechnen

`GlobalAveragePooling2D` gibt bei VGG19 einen Vektor der Länge **512** aus
(Anzahl der Filter im letzten Conv-Block).

Für jeden Dense-Layer gilt:
`Parameter = Eingaben × Neuronen + Neuronen (Biases)`

In [ ]:
# Neuer Kopf: GlobalAvgPool(512) → Dense(128) → Dense(1)
params_dense1 = 512 * 128 + 128
params_dense2 = 128 * 1   + 1
total_head    = params_dense1 + params_dense2

print("=== Manuelle Parameterberechnung: Neuer Kopf ===")
print(f"Dense(128):  512 × 128 + 128 = {params_dense1:>8,}")
print(f"Dense(1):    128 ×   1 +   1 = {params_dense2:>8,}")
print(f"{'─' * 42}")
print(f"Kopf gesamt:                   {total_head:>8,}")
print()

total_params     = model_tl.count_params()
trainable_params = sum(tf.size(w).numpy() for w in model_tl.trainable_weights)
frozen_params    = total_params - trainable_params

print("=== Parametervergleich (gesamt) ===")
print(f"Gesamt-Parameter:        {total_params:>12,}")
print(f"Trainierbare Parameter:  {trainable_params:>12,}  ({trainable_params/total_params*100:.1f} %)")
print(f"Eingefrorene Parameter:  {frozen_params:>12,}  ({frozen_params/total_params*100:.1f} %)")
print()
print(f"Prüfung: manuell = {total_head:,}  ≈  automatisch = {trainable_params:,}  ✓")
print()
print("Fazit: Wir würden weniger als 1 % der Parameter trainieren –")
print("       und könnten trotzdem sehr gute Ergebnisse erzielen!")

# Schritt 12: Modell kompilieren
## Bereit für das Training

Das Modell ist kompiliert und bereit. Für ein echtes Training würden
Trainingsdaten in der Ordnerstruktur `train/teddy/` und `train/no_teddy/`
benötigt.

- `binary_crossentropy` als Loss (Binärklassifikation)
- Niedrige Lernrate `1e-4` – da nur der kleine Kopf trainiert wird

In [ ]:
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
print("Modell kompiliert ✓")
print("Bereit für model.fit() – sobald Trainingsdaten vorhanden sind.")

# Schritt 13: Arbeitsauftrag
## Reflexionsfragen – bitte schriftlich beantworten

1. **Warum ist `include_top=False` so wichtig für Transfer Learning?**
   Tipp: Was enthält der „Top" – und für welche Aufgabe wurde er trainiert?

2. **Was bewirkt das Einfrieren der Gewichte (`layer.trainable = False`)?**
   Was würde passieren, wenn wir die Basis **nicht** einfrieren?

3. **Wie viele Parameter werden tatsächlich trainiert vs. wie viele gibt es insgesamt?**
   Interpretieren Sie das Ergebnis aus Schritt 11.

# Schritt 15: Merksatz
## Merksatz

**Transfer Learning ist heute der Standard in der Praxis –
niemand trainiert große Netze von Grund auf.**

- `include_top=False` trennt Feature-Extraktor vom Klassifikationskopf
- Einfrieren verhindert Catastrophic Forgetting
- Wir trainieren < 1 % der Parameter und erzielen trotzdem hohe Accuracy
- Training dauert Minuten statt Wochen

Ausblick: Im nächsten Notebook (InceptionV3) vergleichen wir eine
effizientere Architektur mit weniger Parametern bei ähnlicher Leistung.